In [ ]:
!pip install -q fastapi uvicorn transformers accelerate bitsandbytes pyngrok nest_asyncio

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-8B"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto"
)
print("Modèle prêt")

In [ ]:
from typing import List
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from google.colab import userdata

API_KEY = userdata.get("API_KEY")

SYSTEM = """You are a medical orientation assistant for a clinic.
Answer in the language of the user, in simple and warm words.

Structure of every answer:
1. One or two sentences showing you understood the symptoms.
2. A short general explanation: common reasons for this kind of symptom.
   Say clearly that only a doctor can confirm.
3. 2 or 3 short questions (duration, intensity from 1 to 10, other signs).
4. Safe advice: rest, drink water, and when to see a doctor quickly.
5. Suggest booking an appointment.

Strict rules:
- Never give a diagnosis. Never say "you have X".
- NEVER mention any medicine, drug, painkiller, antibiotic or dose. Not even paracetamol.
- If there are emergency signs (chest pain, breathing trouble, fainting, heavy bleeding,
  stroke signs, suicidal thoughts), tell the person to call emergency services now.
  Do not ask questions in that case.

Tunisian Arabic is often written with Latin letters and numbers (3=ع, 7=ح, 9=ق, 5=خ).
Examples: weja3=pain, ras=head, skhana=fever, sder=chest, battn=stomach,
ghathayen=nausea, nta3ab=I am tired, ma n9darech nneffes=I cannot breathe,
3andi=I have, men yomein=for two days, lbareh=yesterday.
Understand it. Reply in simple Arabic or French.
If you truly do not understand, ask the person to rephrase. Do not guess."""
print("ok")

app = FastAPI()

class Msg(BaseModel):
    role: str
    content: str

class Req(BaseModel):
    messages: List[Msg]

@app.get("/health")
def health():
    return {"ok": True, "model": MODEL_ID}

@app.post("/chat")
def chat(req: Req, x_api_key: str = Header(None)):
    if x_api_key != API_KEY:
        raise HTTPException(401, "bad key")
    msgs = [{"role": "system", "content": SYSTEM}] + [m.model_dump() for m in req.messages]
    text = tok.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=400,
            do_sample=True, temperature=0.4, top_p=0.9,
        )
    answer = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return {"answer": answer}

In [ ]:
import threading, time, nest_asyncio, uvicorn
from google.colab import userdata
from pyngrok import ngrok

nest_asyncio.apply()
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True
).start()
time.sleep(3)

print(ngrok.connect(8000).public_url)

In [ ]:
import json, re
from typing import Optional

class GenReq(BaseModel):
    system: str
    user: str
    json_schema: Optional[dict] = None

@app.post("/generate")
def generate(req: GenReq, x_api_key: str = Header(None)):
    if x_api_key != API_KEY:
        raise HTTPException(401, "bad key")
    system = req.system
    if req.json_schema:
        system += ("\n\nReturn ONLY one JSON object that matches this JSON schema, "
                   "with no text around it:\n" + json.dumps(req.json_schema))
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": req.user}]
    text = tok.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=800, do_sample=False)
    content = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r"\{.*\}", content, re.S)
    if m:
        content = m.group(0)
    return {"content": content, "model": MODEL_ID}
print("route /generate ajoutée")